## Minimal example with custom CNECs

In [ ]:
import pypsa
import fbmc  # registers pypsa.Network.fbmc accessor

from fbmc.settings import FBMCConfig

We need to set config.cnec_setting to CUSTOM

In [2]:
from fbmc.enums import CNECStrategy
config = FBMCConfig.from_base_yaml("config/base_config.yaml")
config.cnec_setting = CNECStrategy.CUSTOM


Create a simple three-node network (same as in minimal example)

In [3]:
nodal_net = pypsa.Network()
nodal_net.set_snapshots(['1', '2'])
nodal_net.add('Bus', ['A1', 'B1', 'B2'])
nodal_net.buses.loc[:, 'zone_name'] = ['A', 'B', 'B']
nodal_net.add('Line', 'B1-A1', bus0='B1', bus1='A1', x=1, s_nom=12)
nodal_net.add('Line', 'B1-B2', bus0='B1', bus1='B2', x=1, s_nom=12)
nodal_net.add('Line', 'A1-B2', bus0='A1', bus1='B2', x=1, s_nom=12)

nodal_net.add('Generator', 'gen_A1', bus='A1', p_nom=100, marginal_cost=400, carrier="Oil")
nodal_net.add('Generator', 'gen_B1', bus='B1', p_nom=100, marginal_cost=100, carrier="Wind")
nodal_net.add('Generator', 'gen_B2', bus='B2', p_nom=100, marginal_cost=200, carrier="CCGT")
nodal_net.add('Load', 'load_A1', bus='A1', p_set=[15, 15])


Index(['load_A1'], dtype='object')

In [ ]:
zonal_net = nodal_net.fbmc.to_zonal(nodal_net.buses['zone_name'])

### Case without security constraints

Without security constraints, CNECs (more accurately termed CNEs in this case) are defined by a sequence of tuples like \
[ \
    (branch_type, branch_name), \
    ... \
] \
Branch type can be "Line" or "Transformer" (if transformers are present in the network).

We'll selec all lines as CNECs for now. 

In [5]:

line_inds = nodal_net.lines.index
cnecs = [(("Line", line0))
         for line0 in line_inds]
cnecs

[('Line', 'B1-A1'), ('Line', 'B1-B2'), ('Line', 'A1-B2')]

We explicitly pass the CNECs to run_fbmc

In [ ]:
zonal_net.fbmc.create_model(nodal_net, config=config, cnecs=cnecs)
zonal_net.model.solve(**config.solver_kwargs)
fbmc_result = zonal_net.fbmc.results()

### Case with security constraints


In [7]:
config.add_security_constraints = True 

CNECs are now defined by a sequence of tuples like \
[ \
    ((branch_type, branch_name), (contingency_branch_type, contingency_branch_name)), \
    ... \
]

We'll selec all combinations of CNEs and contingencies for now. 

In [8]:
line_inds = nodal_net.lines.index
cnecs = [(("Line", line0), ("Line", line1))
         for line0 in line_inds for line1 in line_inds if line0 != line1]
cnecs

[(('Line', 'B1-A1'), ('Line', 'B1-B2')),
 (('Line', 'B1-A1'), ('Line', 'A1-B2')),
 (('Line', 'B1-B2'), ('Line', 'B1-A1')),
 (('Line', 'B1-B2'), ('Line', 'A1-B2')),
 (('Line', 'A1-B2'), ('Line', 'B1-A1')),
 (('Line', 'A1-B2'), ('Line', 'B1-B2'))]

Re-initialize zonal network since previous optimization was not N-1 line outage security constrained. 

In [ ]:
zonal_net = nodal_net.fbmc.to_zonal(nodal_net.buses['zone_name'])

In [ ]:
zonal_net.fbmc.create_model(nodal_net, config=config, cnecs=cnecs)
zonal_net.model.solve(**config.solver_kwargs)
fbmc_result = zonal_net.fbmc.results()

In [11]:

print(fbmc_result.dispatch_results)
print(fbmc_result.net_positions)

DispatchResult object with attrs: 
  generators_p: (2, 3) snapshots x generators, 
  storage_units_p: (2, 0) snapshots x storage units, 
  links_p0: (2, 0) snapshots x links, 
  storage_levels: None snapshots x storage levels, 
  water_values: None snapshots x water values
<xarray.DataArray 'Zone-p' (snapshot: 2, Zone: 2)> Size: 32B
array([[-12.,  12.],
       [-12.,  12.]])
Coordinates:
  * snapshot  (snapshot) object 16B '1' '2'
  * Zone      (Zone) <U1 8B 'A' 'B'
